# 2. SQL para exploración de datos (Data Discovery)
____
Se debe importar el dataset usando pandas, se escriben las consultas en el archivo `02_sql_discovery.sql` y se cargan a este notebook de jupyter como una lista de queries.

In [79]:
import sqlite3
import pandas as pd
import csv

ventas_techventas = "../data/ventas_techventas.csv"
codificacion = "latin-1"

dataFrame = pd.read_csv(ventas_techventas, encoding=codificacion, quoting=csv.QUOTE_NONE)

# El el dataset, el nombre de Ana López tiene un error de codificación, por lo que se reemplaza al cargarlo
# De igual manera, se eliminan las comillas que hay en algunos de los registros y en algunos de los productos
dataFrame["vendedor"] = dataFrame["vendedor"].str.replace("Ana Lï¿½ï¿", "Ana López")
dataFrame['order_id'] = dataFrame['order_id'].str.strip('"')
dataFrame['vendedor'] = dataFrame['vendedor'].str.strip('"')
dataFrame['producto'] = dataFrame['producto'].str.replace('""', '"')

# Creación de la base de datos sqlite en la memoria
connection = sqlite3.connect(":memory:")

filas_insertadas = dataFrame.to_sql("ventas_techventas", connection, if_exists="replace", index=False)
print(f"Filas insertadas desde el dataset a la base de datos sqlite: {filas_insertadas}")

Filas insertadas desde el dataset a la base de datos sqlite: 60


In [80]:
# Aplicar valor absoluto a la columna cantidad para los valores negativos
_ = dataFrame["cantidad"].abs()

### Importación del archivo .sql
____

In [81]:
def cargar_queries_sql(ruta_de_archivo: str) -> list:
    with open(ruta_de_archivo) as archivo:
        contenido = archivo.read()
        queries = []

        # Se separa el archivo por queries a partir del caracter ;
        for query in contenido.split(";"):
            if query.strip():
                queries.append(query)

        return queries

In [82]:
queries_sql = "../sql/02_sql_discovery.sql"
queries = cargar_queries_sql(queries_sql)

### Query 1: SELECT DISTINCT
____
Se deben listar todos los productos únicos disponibles con un alias para la columna

In [83]:
productos_unicos = pd.read_sql_query(queries[0], connection)
print(productos_unicos)

    productos_unicos
0      Laptop Pro 15
1  Mouse Inalambrico
2        Monitor 27"
3   Teclado Mecanico
4            SSD 1TB
5      Router WiFi 6
6                NaN


### Query 2: WHERE + BETWEEN
----
Se filtran los pedidos del primer trimestre (enero a mar 2024) cuya cantidad sea mayor a 5 unidades

In [84]:
pedidos_filtrados = pd.read_sql_query(queries[1], connection)
print(pedidos_filtrados)

   order_id       fecha  cantidad      vendedor
0      1002  2024-01-07        15   Carlos Ruiz
1      1004  2024-01-12        10     Luis Mora
2      1006  2024-01-18        20  Maria Torres
3      1008  2024-01-22         8     Luis Mora
4      1009  2024-01-25        12  Maria Torres
5      1012  2024-02-05        30  Maria Torres
6      1015  2024-02-12        25  Maria Torres
7      1017  2024-02-18         6     Luis Mora
8      1018  2024-02-20        18   Carlos Ruiz
9      1019  2024-02-22        12  Maria Torres
10     1022  2024-03-04        30   Carlos Ruiz
11     1023  2024-03-07         8  Maria Torres
12     1026  2024-03-15        20   Carlos Ruiz
13     1027  2024-03-18        15  Maria Torres
14     1028  2024-03-20         6     Ana López
15     1030  2024-03-25        10   Carlos Ruiz


### Query 3: GROUP BY + HAVING
----
Se muestran los vendedores cuyo ingreso bruto total supera los 10 mil dólares

In [85]:
vendedores = pd.read_sql_query(queries[2], connection)
print(vendedores)

       vendedor  ingreso_bruto_total
0     Ana López              28510.0
1   Carlos Ruiz              21355.0
2     Luis Mora              21930.0
3  Maria Torres              16410.0


### Query 4: COUNT + SUM + AVG
----
Se muestra el total de pedidos, suma de unidades y precio unitario promedio de cada categoría

In [86]:
filtro_categorias = pd.read_sql_query(queries[3], connection)
print(filtro_categorias)

        categoria  numero_pedidos  unidades_vendidas  precio_unitario_promedio
0  Almacenamiento              12                260                      95.0
1     Electronica              25                 72                     792.0
2     Perifericos              15                215                      53.0
3           Redes               8                 39                     120.0


### QUERY 5: JOIN
----
Crear una tabla para los vendedores, y calcular posteriormente el porcentaje de cumplimiento de la meta mensual de cada uno

In [87]:
query_tabla_vendedores = """--sql
    CREATE TABLE "vendedores" (
        vendedor TEXT,
        zona TEXT,
        meta_mensual INT
    );
"""

query_insertar_datos = """--sql
    INSERT INTO "vendedores" (vendedor, zona, meta_mensual) VALUES 
    ('Ana López', 'Norte', 8000),
    ('Carlos Ruiz', 'Sur', 7500),
    ('Luis Mora', 'Norte', 8000),
    ('Maria Torres', 'Centro', 7000);
"""

cursor = connection.cursor()

cursor.execute(query_tabla_vendedores)
cursor.execute(query_insertar_datos)
connection.commit()

#### Cálculo del porcentaje de cumplimiento por mes

In [88]:
cumplimiento_vendedores = pd.read_sql_query(queries[4], connection)
print(cumplimiento_vendedores)

        vendedor mes  meta_mensual  ingresos_brutos  porcentaje_cumplimiento
0      Ana López  01          8000           4850.0                60.625000
1    Carlos Ruiz  01          7500           5175.0                69.000000
2      Luis Mora  01          8000           1810.0                22.625000
3   Maria Torres  01          7000           3040.0                43.428571
4      Ana López  02          8000           6200.0                77.500000
5    Carlos Ruiz  02          7500           2070.0                27.600000
6      Luis Mora  02          8000           2525.0                31.562500
7   Maria Torres  02          7000           3425.0                48.928571
8      Ana López  03          8000           1770.0                22.125000
9    Carlos Ruiz  03          7500           4200.0                56.000000
10     Luis Mora  03          8000           6480.0                81.000000
11  Maria Torres  03          7000           3155.0                45.071429

### Query 6: SUBCONSULTA
----
Encontrar el cliente que generó el ingreso total en el primer semestre

In [89]:
cliente_mayor = pd.read_sql_query(queries[5], connection)
print(cliente_mayor)

  cliente_id  ingreso_total
0       C001        20425.0


In [90]:
connection.close()